# 03 – BM25-Retrieval pro Session

Dieses Notebook führt **BM25-Retrieval** für jede Session durch und extrahiert
pseudo-relevante sowie nicht-relevante Dokumente als Grundlage für die TF-IDF-Evidence.

**Retrieval-Strategie:**
- `rel` (pseudo-relevant): Top-3 Treffer (Rang 1–3)
- `non_rel` (nicht-relevant): Treffer an Rang 100–103 (thematisch entfernt, aber im Korpus)

Queries mit weniger als 103 Treffern werden übersprungen und in Cell 5 manuell nachgepflegt.

**Voraussetzungen:** `01_indexierung.ipynb` und `02_sessions.ipynb` wurden ausgeführt.

**Ausgabe:** `data/retrieval_bm25_topic.csv`

## 1. Imports

In [4]:
import pandas as pd
import pyterrier as pt
from pathlib import Path

## 2. Sessions laden

Liest die in `02_sessions.ipynb` erstellte Session-CSV ein.

In [5]:
# Sessions laden
sessions_df = pd.read_csv("data/sessions.csv")
print("Geladene Sessions:", len(sessions_df))

Geladene Sessions: 381


## 3. PyTerrier initialisieren und BM25-Retriever laden

Lädt den in `01_indexierung.ipynb` erstellten Index und konfiguriert
den BM25-Retriever mit direktem Zugriff auf `docno` und `text`.

In [7]:
# PyTerrier initialisieren und BM25-Retriever auf dem gespeicherten Index aufbauen
if not pt.java.started():
    pt.java.init()

index_path = str(Path.cwd() / "Index_Longeval_SCI_snapshot3")
index = pt.IndexFactory.of(index_path)

# metadata=["docno", "text"] stellt sicher, dass Dokumenttexte direkt verfügbar sind
bm25 = pt.terrier.Retriever(index, wmodel="BM25", metadata=["docno", "text"])

Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


16:59:00.603 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1,3 GiB of memory would be required.


## 4. Retrieval-Loop

Für jede Session werden Top-103 Treffer abgerufen.

Sonderzeichen in den Query-Texten werden vorab bereinigt,
da Terrier bei Zeichen wie `?`, `'`, `^` oder `"` Parse-Fehler wirft.

Pro Session entstehen zwei Zeilen in der Ausgabe-CSV:
eine mit den relevanten (`rel`) und eine mit den nicht-relevanten (`non_rel`) Dokumenten.

In [8]:
# BM25-Retrieval: Top-103 Treffer pro Session
# Positionen 0-2 = pseudo-relevant, 99-102 = nicht-relevant
import re

TOP_K = 103
REL_N = 3          # Anzahl relevanter Dokumente (Rang 1-3)
NONREL_START = 99  # Start der nicht-relevanten Dokumente
NONREL_END = 103   # Ende der nicht-relevanten Dokumente

# Terrier reagiert auf Sonderzeichen mit Parse-Fehlern → Bereinigung nötig
def clean_for_terrier(q: str) -> str:
    if q is None:
        return ""
    q = str(q)
    # Entfernt alle Zeichen, die Terriers Query-Parser stören
    for char in [";", "'", "\u2019", "?"]:
        q = q.replace(char, " ")
    q = re.sub(r'[\^"\\{}\[\]\(\)[:+\-!~*?#@$%&|/\<>]', " ", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q

rows = []

for _, s in sessions_df.iterrows():
    sid = s["session_id"]
    query_text = clean_for_terrier(s["first_query"])

    res = bm25.search(query_text).head(TOP_K)

    # Sessions mit zu wenigen Treffern überspringen (werden in Cell 5 manuell ergänzt)
    if len(res) < NONREL_END:
        print(f"Query {sid}: nur {len(res)} Treffer, uebersprungen")
        continue

    docnos = res["docno"].astype(str).tolist()
    texts = res["text"].astype(str).tolist()

    rows.append({
        "session_id": sid,
        "doc_variant": "rel",
        "query_text": query_text,
        "docnos": docnos[:REL_N],
        "texts": texts[:REL_N],
    })

    rows.append({
        "session_id": sid,
        "doc_variant": "non_rel",
        "query_text": query_text,
        "docnos": docnos[NONREL_START:NONREL_END],
        "texts": texts[NONREL_START:NONREL_END],
    })

retrieval_df = pd.DataFrame(rows)
retrieval_df.to_csv("data/retrieval_bm25_topic.csv", index=False)

print("Anzahl Retrieval-Zeilen:", len(retrieval_df))

Query 547f504ff3c897aa262396385d4af920: nur 4 Treffer, uebersprungen
Query e777fb4e6e6a57942bf45de4eb297a56: nur 2 Treffer, uebersprungen
Query a76f803a54168ee973b5efb8ec3eff57: nur 46 Treffer, uebersprungen
Query a681a45c56438f13b7dfeca913f133e3: nur 58 Treffer, uebersprungen
Query 7943fb88b63cdf0a8aaff0b67e709cd6: nur 30 Treffer, uebersprungen
Anzahl Retrieval-Zeilen: 752


## 5. Manuelle Einträge für fehlgeschlagene Sessions

Einige Queries (z.B. `"obezite"`, `"as-ack"`, `"kondraske"`, `"cervids"`, `"yoshua bengio"`)
liefern weniger als 103 BM25-Treffer und wurden im Loop übersprungen.
Diese Sessions erhalten manuell erstellte TREC-Topics,
die direkt an die bestehenden JSONL-Runfiles angehängt werden.

In [9]:
# Manuelle Topics für Sessions mit zu wenigen BM25-Treffern
import json
import os

manual_entries = [
    {
        "qid": "547f504ff3c897aa262396385d4af920",
        "query": "yoshua bengio",
        "description": "Documents related to the work and contributions of Yoshua Bengio, a prominent researcher in artificial intelligence and deep learning.",
        "narrative": "Relevant documents include research papers authored or co-authored by Yoshua Bengio, publications on deep learning and neural networks, and citations of his academic impact. Documents unrelated to Bengio or deep learning are not relevant."
    },
    {
        "qid": "e777fb4e6e6a57942bf45de4eb297a56",
        "query": "kondraske",
        "description": "Documents related to research or work associated with Kondraske.",
        "narrative": "Relevant documents include academic publications or research associated with Kondraske. Documents unrelated to this researcher are not relevant."
    },
    {
        "qid": "a76f803a54168ee973b5efb8ec3eff57",
        "query": "as-ack",
        "description": "Documents related to the AS-ACK protocol or acknowledgment mechanism in network communication.",
        "narrative": "Relevant documents include technical descriptions of the AS-ACK protocol and research on acknowledgment mechanisms."
    },
    {
        "qid": "a681a45c56438f13b7dfeca913f133e3",
        "query": "cervids",
        "description": "Documents related to cervids, the family of mammals including deer, elk, and moose.",
        "narrative": "Relevant documents include biological and ecological research on cervid species."
    },
    {
        "qid": "7943fb88b63cdf0a8aaff0b67e709cd6",
        "query": "obezite",
        "description": "Documents related to obesity research, particularly from Turkish-language literature.",
        "narrative": "Relevant documents include medical research on obesity causes, treatment, and prevention."
    }
]

os.makedirs("runfiles", exist_ok=True)

# Manuelle Einträge an beide Runfile-Varianten anhängen
for variant in ["rel", "non_rel"]:
    path = f"runfiles/openai_first_{variant}_tfidf5.jsonl"
    with open(path, "a", encoding="utf-8") as f:
        for entry in manual_entries:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"Hinzugefuegt zu {path}")

# Gesamtanzahl Einträge prüfen
for variant in ["rel", "non_rel"]:
    path = f"runfiles/openai_first_{variant}_tfidf5.jsonl"
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"{path}: {len(lines)} Eintraege")

Hinzugefuegt zu runfiles/openai_first_rel_tfidf5.jsonl
Hinzugefuegt zu runfiles/openai_first_non_rel_tfidf5.jsonl
runfiles/openai_first_rel_tfidf5.jsonl: 386 Eintraege
runfiles/openai_first_non_rel_tfidf5.jsonl: 386 Eintraege
